In [1]:
import re

import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

import torch
import numpy as np
import evaluate

from sklearn.model_selection import train_test_split

from torch.utils.data import DataLoader, TensorDataset

C:\Natural Language Processing\Entity Sentiment Analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
df = pd.read_csv("../../datasets/twitter 3m rows/twitter_dataset.csv")

In [4]:
df.head()

,Unnamed: 0,tweet,sentiment
0,0,is upset that he can't update his Facebook by ...,0.0
1,1,@Kenichan I dived many times for the ball. Man...,0.0
2,2,my whole body feels itchy and like its on fire,0.0
3,3,"@nationwideclass no, it's not behaving at all....",0.0
4,4,@Kwesidei not the whole crew,0.0


In [5]:
df = df.drop(columns=["Unnamed: 0"])

In [6]:
df.head()

,tweet,sentiment
0,is upset that he can't update his Facebook by ...,0.0
1,@Kenichan I dived many times for the ball. Man...,0.0
2,my whole body feels itchy and like its on fire,0.0
3,"@nationwideclass no, it's not behaving at all....",0.0
4,@Kwesidei not the whole crew,0.0


In [7]:
def preprocess_tweet(text):
    # Replace URLs
    text = re.sub(r"http\S+|www\.\S+", "HTTPURL", text)
    # Replace @mentions
    text = re.sub(r"@\w+", "@USER", text)
    return text

In [8]:
df["tweet"] = df["tweet"].apply(preprocess_tweet)

In [9]:
df = df.drop_duplicates()
df = df.dropna(subset=["sentiment"])

In [10]:
df.isna().sum()

tweet        0
sentiment    0
dtype: int64

In [11]:
df["sentiment"] = df["sentiment"].astype(int)

In [12]:
df.head()

,tweet,sentiment
0,is upset that he can't update his Facebook by ...,0
1,@USER I dived many times for the ball. Managed...,0
2,my whole body feels itchy and like its on fire,0
3,"@USER no, it's not behaving at all. i'm mad. w...",0
4,@USER not the whole crew,0


In [13]:
MAX_ROWS = 300000

In [14]:
df_compressed, _ = train_test_split(df, train_size=MAX_ROWS, stratify=df['sentiment'], random_state=42)

In [15]:
df_train, df_val =  train_test_split(df_compressed, test_size=0.2, stratify=df_compressed['sentiment'], random_state=42)

In [16]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [17]:
hf_train = Dataset.from_pandas(df_train)
hf_val = Dataset.from_pandas(df_val)

In [18]:
print(hf_train, hf_val)

Dataset({
    features: ['tweet', 'sentiment', '__index_level_0__'],
    num_rows: 240000
}) Dataset({
    features: ['tweet', 'sentiment', '__index_level_0__'],
    num_rows: 60000
})


In [19]:
hf_train = hf_train.rename_columns({"tweet" : "text", "sentiment" : "label"})
hf_train = hf_train.remove_columns("__index_level_0__")
print(hf_train)

Dataset({
    features: ['text', 'label'],
    num_rows: 240000
})


In [20]:
hf_val = hf_val.rename_columns({"tweet" : "text", "sentiment" : "label"})
hf_val = hf_val.remove_columns("__index_level_0__")
print(hf_val)

Dataset({
    features: ['text', 'label'],
    num_rows: 60000
})


In [21]:
tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_val = hf_val.map(tokenize_function, batched=True)

Map: 100%|██████████| 60000/60000 [00:06<00:00, 9901.60 examples/s] 


In [22]:
print(tokenized_train)
print(tokenized_val)

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 240000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 60000
})


In [23]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [24]:
MODEL_SAVE_PATH = "../../models/bert-base-uncased-sentiment-3m-finetuned-154m"

In [25]:
training_args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    eval_strategy="steps",  # eval lebih sering, bukan cuma tiap epoch
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,  # simpan max 2 checkpoint, hemat storage
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,  # ~1700 steps warmup, bantu epoch 1
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,  # tambahkan ini — kasih tau "accuracy makin tinggi makin bagus"
)

In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [ ]:
print("Start training")
trainer.train()

Start training


Step,Training Loss,Validation Loss,Accuracy
500,0.964700,0.749118,0.632050
1000,0.636200,0.545415,0.761050
1500,0.510600,0.476321,0.796400
2000,0.469000,0.454959,0.805117
2500,0.453900,0.477646,0.800367
3000,0.464800,0.428091,0.814183
3500,0.447400,0.439836,0.816600
4000,0.427000,0.428938,0.815483
4500,0.447000,0.427690,0.814333
5000,0.439200,0.444981,0.817267


In [ ]:
# checkpoint 29500 --> train loss 0.38, val loss 0.4, acc 0.84

# checkpoint 59000--> train loss 0.34, val loss 0.41, acc 0.85

In [ ]:
# trainer.save_model(MODEL_SAVE_PATH)
# tokenizer.save_pretrained(MODEL_SAVE_PATH)

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

C:\Natural Language Processing\Entity Sentiment Analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
BEST_CP = "../../models/bert-base-uncased-3m-checkpoints/checkpoint-59000"

In [3]:
FINAL_MODEL = "../../models/bert-uncased-3m"

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(BEST_CP)
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

In [8]:
model.save_pretrained(FINAL_MODEL)
tokenizer.save_pretrained(FINAL_MODEL)

('../../models/bert-uncased-3m\\tokenizer_config.json',
 '../../models/bert-uncased-3m\\special_tokens_map.json',
 '../../models/bert-uncased-3m\\vocab.txt',
 '../../models/bert-uncased-3m\\added_tokens.json',
 '../../models/bert-uncased-3m\\tokenizer.json')

In [ ]:
|# MODEL_PATH = "../../models/roberta-sentiment-finetuned-154m"
#
# # 1. load model
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
#
# # 2. metrics
# metric_accuracy = evaluate.load("accuracy")
# metric_f1 = evaluate.load("f1")
#
#
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)
#     accuracy = metric_accuracy.compute(predictions=predictions, references=labels)
#     f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
#     return {"accuracy": accuracy["accuracy"], "f1": f1["f1"]}
#
#
# # 3. evaluate
# trainer = Trainer(model=model, compute_metrics=compute_metrics)
# results = trainer.evaluate(eval_dataset=tokenized_val)
# print(results)